# Incident Response Runbook: Drift Protocol — DPRK Six-Month Operation

**Tactic:** Initial Access → Lateral Movement → Privilege Escalation → Impact
**Technique:** T1566.001 + T1195.001 + T1550 + T1218
**Severity:** CRITICAL

## Overview

This runbook covers the Drift Protocol attack attributed to DPRK actors (April 2026, $285M loss).
The operation unfolded over six months: attackers built rapport with protocol developers at
conferences posing as a quant trading firm, then delivered a malicious VSCode/Cursor repository.
Compromise of developer machines enabled durable nonce pre-signing, governance manipulation,
zero-timelock multisig migration, and oracle price manipulation leading to the largest DeFi
protocol drain to date.

## MITRE ATT&CK Mapping

| Technique | ID | Description |
|---|---|---|
| Spearphishing Attachment | **T1566.001** | Malicious VSCode/Cursor repo sent via trusted conference contact |
| Supply Chain Compromise | **T1195.001** | Backdoored VSCode extension / dev toolchain in shared repo |
| Use Alternate Authentication Material | **T1550** | Durable nonces pre-signed on compromised developer machines |
| Signed Binary Proxy Execution | **T1218** | Legitimate VSCode/Cursor used as malware loader |
| Modify Authentication Process | **T1556** | Multisig timelock removed via governance proposal |
| Financial Theft | **T1657** | Oracle manipulation + multisig admin control → protocol drain |

## Lateral Movement Analysis

This attack demonstrates the most sophisticated DeFi lateral movement chain documented to date:

1. **Conference social engineering** — Attackers (posing as "Nexus Quant") spend 3 months building rapport with Drift developers at DeFi Summit, Breakpoint, and Token2049
2. **Malicious repo delivery** — Developer #1 clones shared "high-frequency trading strategy" VSCode workspace; backdoored `.vscode/extensions` + `settings.json` executes on workspace open
3. **Developer machine #1 compromised** — Keylogger + memory scraper; attacker extracts signing seed phrase and pre-signs durable nonce transactions for future use
4. **Lateral to developer machine #2** — Shared git repository configuration + internal Slack pivot; attacker deploys second-stage payload via CI/CD webhook
5. **Governance signer access** — Two of five governance multisig signers are now compromised
6. **Governance manipulation** — Malicious proposal submitted to remove timelock (framed as "gas optimization"); passes with 2/5 signer approval
7. **Zero-timelock multisig migration** — Protocol admin control transferred to attacker-controlled multisig
8. **Oracle manipulation + drain** — Attacker manipulates DRIFT/USDC oracle, opens max leverage positions, drains protocol insurance fund

**Full lateral movement chain:**
`conference trust → VSCode repo → dev machine #1 → CI/CD webhook → dev machine #2 → 2/5 governance signers → timelock removal → multisig admin control → oracle manipulation → $285M drain`

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Identify which developer machines were compromised and when
- Reconstruct the governance manipulation timeline
- Audit all durable nonce accounts for pre-signed unauthorized transactions
- Determine full scope of oracle manipulation and position drain


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

print("=" * 60)
print("STEP 1: Detection & Analysis — Drift Protocol DPRK Operation")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
splunk_indicators = []
unique_users = set()
source_hosts = set()

# Detect anomalous VSCode extension / workspace activity
print("\n[QUERY] Detecting anomalous VSCode workspace and extension activity...")
vscode_query = '''
index=endpoint OR index=sysmon
(process_name="code" OR process_name="cursor") 
(child_process_name="python*" OR child_process_name="node" OR child_process_name="curl" OR child_process_name="wget")
NOT parent_command_line="*extensions.json*"
| stats count by host, process_name, child_process_name, command_line, user, _time
| where count > 0
'''
try:
    vscode_results = splunk.search_events(vscode_query, timeframe="-180d")
    print(f"   Found {len(vscode_results)} anomalous VSCode child process events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    vscode_results = []

for event in vscode_results:
    affected_systems.append({'hostname': event.get('host', 'unknown'), 'process': event.get('child_process_name', ''), 'user': event.get('user', ''), 'last_seen': event.get('_time', detection_time)})
    unique_users.add(event.get('user', 'unknown'))
    source_hosts.add(event.get('host', 'unknown'))
    splunk_indicators.append({'type': 'vscode_anomaly', 'value': f"host={event.get('host')} child={event.get('child_process_name')} cmd={event.get('command_line','')[:80]}", 'context': 'VSCode spawning unexpected child process — potential backdoored extension'})

# Detect durable nonce account creation by governance signers
print("\n[QUERY] Auditing durable nonce accounts created by governance signers...")
nonce_query = '''
index=onchain_events
instruction="InitializeNonceAccount" OR instruction="AdvanceNonceAccount"
| lookup governance_signer_addresses as signer
| where isnotnull(governance_role)
| stats count, values(nonce_account) as nonce_accounts by signer, governance_role, _time
| sort -count
'''
try:
    nonce_results = splunk.search_events(nonce_query, timeframe="-180d")
    print(f"   Found {len(nonce_results)} durable nonce creation events from governance signers")
    for r in nonce_results:
        splunk_indicators.append({'type': 'durable_nonce_creation', 'value': f"signer={r.get('signer','?')[:16]} role={r.get('governance_role')} nonces={r.get('nonce_accounts')}", 'context': 'Governance signer created durable nonce — potential pre-signing for unauthorized use'})
except Exception as e:
    print(f"   Nonce audit query failed: {e}")

# Detect governance proposal with timelock removal
print("\n[QUERY] Auditing governance proposals for timelock or authority changes...")
gov_query = '''
index=onchain_events program="drift_governance"
(instruction="CreateProposal" OR instruction="CastVote" OR instruction="ExecuteTransaction")
| stats values(instruction) as instructions, values(signer) as signers, values(proposal_data) as data by proposal_id, _time
| where data LIKE "*timelock*" OR data LIKE "*setAuthority*" OR data LIKE "*removeDelay*"
'''
try:
    gov_results = splunk.search_events(gov_query, timeframe="-90d")
    print(f"   Found {len(gov_results)} governance proposals touching timelock or authority")
    for r in gov_results:
        splunk_indicators.append({'type': 'governance_manipulation', 'value': f"proposal={r.get('proposal_id','?')} signers={r.get('signers')} data={str(r.get('data',''))[:80]}", 'context': 'Governance proposal modifying timelock or authority — requires immediate review'})
except Exception as e:
    print(f"   Governance audit query failed: {e}")

# Detect oracle price manipulation
print("\n[QUERY] Detecting oracle price anomalies correlated with large position opens...")
oracle_query = '''
index=onchain_events program="drift_oracle" OR program="pyth_network" OR program="switchboard"
| join market [search index=onchain_events program="drift_protocol" instruction="openPosition" leverage > 10]
| eval price_deviation = abs(oracle_price - twap_price) / twap_price * 100
| where price_deviation > 5
| stats max(price_deviation) as max_deviation, sum(position_size) as total_position_usd by market, _time
'''
try:
    oracle_results = splunk.search_events(oracle_query, timeframe="-7d")
    print(f"   Found {len(oracle_results)} oracle manipulation + leverage correlation events")
    for r in oracle_results:
        splunk_indicators.append({'type': 'oracle_manipulation', 'value': f"market={r.get('market')} deviation={r.get('max_deviation'):.1f}% position={r.get('total_position_usd')} USD", 'context': 'Oracle price deviation correlated with max-leverage position open'})
except Exception as e:
    print(f"   Oracle anomaly query failed: {e}")

# CrowdStrike detections on developer machines
print("\n[QUERY] Checking CrowdStrike for developer machine compromise indicators...")
try:
    cs_detections = crowdstrike.get_detections(filter="technique:'Spearphishing' OR technique:'Supply Chain'", start_time="-180d")
    for det in cs_detections:
        affected_systems.append({'hostname': det.get('hostname',''), 'device_id': det.get('device_id',''), 'detection_id': det.get('detection_id',''), 'technique': det.get('technique','')})
    print(f"   CrowdStrike: {len(cs_detections)} spearphishing/supply chain detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")

# MISP enrichment
print("\n[ENRICHMENT] Checking MISP for DPRK Lazarus infrastructure...")
misp_results = []
try:
    dprk_iocs = ["nexus-quant.io", "nexusquant.com"]
    for ioc in dprk_iocs:
        hits = misp.search_iocs(ioc)
        if hits:
            misp_results.extend(hits)
            print(f"   MISP DPRK hit: {ioc} — {len(hits)} events")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

print("\n[CASE] Creating IRIS incident case...")
try:
    incident_id = iris.create_case({
        'title': f'Drift Protocol DPRK Operation — $285M drain — {len(affected_systems)} developer machines',
        'severity': 'CRITICAL',
        'threat_actor': 'Lazarus Group (DPRK)',
        'technique': 'T1566.001 + T1195.001 + T1550 + T1218',
        'indicators': splunk_indicators
    })
    print(f"   IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n✅ Detection complete:")
print(f"   - VSCode anomaly events: {len([i for i in splunk_indicators if i['type']=='vscode_anomaly'])}")
print(f"   - Durable nonce creation events: {len([i for i in splunk_indicators if i['type']=='durable_nonce_creation'])}")
print(f"   - Governance manipulation events: {len([i for i in splunk_indicators if i['type']=='governance_manipulation'])}")
print(f"   - Oracle manipulation events: {len([i for i in splunk_indicators if i['type']=='oracle_manipulation'])}")
print(f"   - Compromised developer machines: {len(affected_systems)}")
print(f"   - DPRK MISP hits: {len(misp_results)}")
print(f"   - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Emergency pause all Drift Protocol operations
- Revoke all governance signer access immediately
- Cancel any pending multisig proposals
- Isolate all compromised developer machines
- Alert oracle providers to halt price updates


In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. Emergency protocol pause
print("\n[CONTAINMENT] Triggering Drift Protocol emergency pause...")
try:
    pause_result = shuffle.call_program_instruction(
        program='drift_protocol',
        instruction='adminPause',
        authority='emergency_multisig',
        note='DPRK incident — emergency pause'
    )
    if pause_result:
        containment_actions.append({'action': 'protocol_pause', 'target': 'drift_protocol all markets', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ Drift Protocol all markets paused")
    else:
        print("   ❌ CRITICAL: Automatic pause failed — manual multisig execution required immediately")
except Exception as e:
    print(f"   Emergency pause failed: {e}")

# 2. Revoke all current governance signers — require full re-key ceremony
print("\n[CONTAINMENT] Revoking all governance signers and cancelling pending proposals...")
try:
    revoke_result = shuffle.revoke_all_governance_signers(program='drift_governance', reason='DPRK compromise')
    if revoke_result:
        containment_actions.append({'action': 'governance_signer_revocation', 'target': 'all 5 governance signers', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ All governance signers revoked")

    cancel_result = shuffle.cancel_all_pending_proposals(program='drift_governance')
    if cancel_result:
        containment_actions.append({'action': 'pending_proposals_cancelled', 'target': 'all pending proposals', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ All pending governance proposals cancelled")
except Exception as e:
    print(f"   Governance revocation failed: {e}")

# 3. Invalidate all outstanding durable nonces
print("\n[CONTAINMENT] Advancing/invalidating all outstanding durable nonce accounts...")
try:
    nonce_accounts = [i.get('value','').split('nonces=')[1] for i in splunk_indicators if i.get('type') == 'durable_nonce_creation' and 'nonces=' in i.get('value','')]
    for nonce_list in nonce_accounts:
        advance_result = shuffle.advance_nonce_account(nonce_list)
        if advance_result:
            containment_actions.append({'action': 'nonce_invalidation', 'target': nonce_list[:40], 'status': 'success', 'timestamp': containment_time})
    print(f"   ✅ Durable nonces advanced/invalidated: {len(nonce_accounts)} accounts")
except Exception as e:
    print(f"   Nonce invalidation failed: {e}")

# 4. Isolate compromised developer machines
print("\n[CONTAINMENT] Isolating compromised developer machines...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.isolate_host(system['device_id'])
        else:
            result = shuffle.isolate_system(system['hostname'])
        if result:
            isolated_hosts.append(system['hostname'])
            containment_actions.append({'action': 'machine_isolation', 'target': system['hostname'], 'status': 'success', 'timestamp': containment_time})
            print(f"   Isolated developer machine: {system['hostname']}")
except Exception as e:
    print(f"   Machine isolation failed: {e}")

# 5. Alert oracle providers
print("\n[CONTAINMENT] Alerting oracle providers to halt price updates...")
oracle_providers = ['pyth_network', 'switchboard', 'chainlink']
try:
    for provider in oracle_providers:
        alert_result = shuffle.send_emergency_alert(provider, 'Drift Protocol DPRK incident — request oracle circuit breaker activation')
        if alert_result:
            containment_actions.append({'action': 'oracle_circuit_breaker_alert', 'target': provider, 'status': 'success', 'timestamp': containment_time})
            print(f"   ✅ Oracle circuit breaker alert sent: {provider}")
except Exception as e:
    print(f"   Oracle alert failed: {e}")

# 6. Block DPRK infrastructure
print("\n[CONTAINMENT] Blocking DPRK attacker infrastructure...")
dprk_domains = ["nexus-quant.io", "nexusquant.com", "drift-governance-api.net"]
try:
    for domain in dprk_domains:
        result = shuffle.block_domain(domain)
        if result:
            containment_actions.append({'action': 'domain_block', 'target': domain, 'status': 'success', 'timestamp': containment_time})
            print(f"   Blocked DPRK domain: {domain}")
except Exception as e:
    print(f"   Domain blocking failed: {e}")

print(f"\n✅ Containment complete:")
print(f"   - Protocol paused: ✓")
print(f"   - Governance signers revoked: ✓")
print(f"   - Pending proposals cancelled: ✓")
print(f"   - Durable nonces invalidated: ✓")
print(f"   - Developer machines isolated: {len(isolated_hosts)}")
print(f"   - Oracle providers alerted: {len(oracle_providers)}")


## Phase 3: Eradication

### Objectives
- Rebuild all developer machines from scratch
- Conduct full Drift Protocol smart contract audit for backdoors
- Restore original governance structure with hardware-only signers
- Remove any persistent access DPRK established in CI/CD or infra


In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []

# 1. Remove backdoored VSCode repo / extensions
print("\n[ERADICATION] Removing backdoored VSCode/Cursor repositories and extensions...")
vscode_cleanup_script = '''
#!/bin/bash
# Remove malicious repo
rm -rf ~/projects/nexus-quant-hft-strategy/
# Remove all non-marketplace extensions
code --list-extensions | xargs -I{} code --uninstall-extension {}
# Audit .vscode/settings.json for malicious configs
find ~ -path "*/.vscode/settings.json" -exec grep -l "terminal.integrated.env\|tasks.json\|launch.json" {} \;
# Clear VSCode workspace state
rm -rf ~/.config/Code/User/workspaceStorage/
rm -rf ~/.cursor/User/workspaceStorage/
'''
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.run_script(system['device_id'], vscode_cleanup_script)
            if result:
                eradication_actions.append({'action': 'vscode_cleanup', 'target': system['hostname'], 'status': 'success', 'timestamp': eradication_time})
                print(f"   VSCode cleaned: {system['hostname']}")
except Exception as e:
    print(f"   VSCode cleanup failed: {e}")

# 2. Full smart contract audit for backdoors
print("\n[ERADICATION] Initiating full Drift Protocol smart contract audit...")
try:
    audit_request = shuffle.create_security_audit_request(
        target='drift_protocol',
        scope=['all_program_accounts', 'governance_logic', 'oracle_integration', 'multisig_config', 'timelock_logic'],
        urgency='emergency',
        auditors=['ottersec', 'neodyme', 'halborn']
    )
    eradication_actions.append({'action': 'smart_contract_audit', 'target': 'drift_protocol full program', 'status': 'initiated', 'timestamp': eradication_time})
    print("   ✅ Emergency smart contract audit initiated with 3 auditor firms")
except Exception as e:
    print(f"   Audit initiation failed: {e}")

# 3. Audit and clean CI/CD systems
print("\n[ERADICATION] Auditing CI/CD pipelines for DPRK persistence...")
cicd_audit_query = '''
index=github_audit OR index=ci_logs
(event_type="webhook_added" OR event_type="secret_added" OR event_type="runner_registration")
date > "2025-10-01"
| stats count by actor, event_type, repo, _time
| sort -_time
'''
try:
    cicd_results = splunk.search_events(cicd_audit_query, timeframe="-180d")
    print(f"   Found {len(cicd_results)} CI/CD changes in the compromise window requiring audit")
    for r in cicd_results:
        eradication_actions.append({'action': 'cicd_finding', 'target': f"{r.get('repo','?')}:{r.get('event_type','?')}", 'status': 'requires_review', 'timestamp': eradication_time})
        print(f"   ⚠️  Review: {r.get('event_type')} on {r.get('repo','?')} by {r.get('actor','?')}")
except Exception as e:
    print(f"   CI/CD audit failed: {e}")

# 4. Rotate all signing infrastructure
print("\n[ERADICATION] Rotating all Drift Protocol signing infrastructure...")
signing_components = ['governance_multisig', 'oracle_authority', 'insurance_fund_authority', 'fee_authority', 'upgrade_authority']
try:
    for component in signing_components:
        rotate_result = shuffle.rotate_protocol_authority('drift_protocol', component, new_type='hardware_wallet_multisig')
        if rotate_result:
            eradication_actions.append({'action': 'authority_rotation', 'target': component, 'status': 'success', 'timestamp': eradication_time})
            print(f"   Rotated {component} to hardware wallet multisig")
except Exception as e:
    print(f"   Authority rotation failed: {e}")

print(f"\n✅ Eradication complete:")
print(f"   - VSCode environments cleaned: ✓")
print(f"   - Smart contract audit initiated: ✓")
print(f"   - CI/CD findings for review: {len([a for a in eradication_actions if a.get('action')=='cicd_finding'])}")
print(f"   - Authorities rotated: {len([a for a in eradication_actions if a.get('action')=='authority_rotation'])}")


## Phase 4: Recovery

### Objectives
- Deploy verified clean Drift Protocol program with 72h timelock restored
- Implement oracle circuit breakers
- Resume protocol operations under hardened security posture
- Establish real-time governance monitoring


In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
restored_services = []

# 1. Deploy audited program with timelock restored
print("\n[RECOVERY] Deploying audited Drift Protocol with governance hardening...")
try:
    deploy_result = shuffle.deploy_program_upgrade(
        'drift_protocol',
        config={
            'governance_timelock_seconds': 259200,  # 72 hours
            'multisig_threshold': '4/7',
            'all_signers_hardware_only': True,
            'upgrade_authority': 'hardware_multisig_only',
            'oracle_circuit_breaker_enabled': True,
            'max_oracle_deviation_pct': 2.0,
            'durable_nonce_monitoring': True
        }
    )
    if deploy_result:
        restored_services.append('drift_protocol v2')
        recovery_actions.append({'action': 'clean_program_deploy', 'target': 'drift_protocol audited + hardened', 'status': 'success', 'timestamp': recovery_time})
        print("   ✅ Audited Drift Protocol deployed with 72h timelock + oracle circuit breakers")
except Exception as e:
    print(f"   Program deployment failed: {e}")

# 2. Configure oracle circuit breakers
print("\n[RECOVERY] Configuring oracle circuit breakers...")
try:
    for provider in ['pyth_network', 'switchboard']:
        cb_result = shuffle.configure_oracle_circuit_breaker(
            provider=provider,
            protocol='drift_protocol',
            max_deviation_pct=2.0,
            max_staleness_seconds=30,
            halt_on_breach=True
        )
        if cb_result:
            recovery_actions.append({'action': 'oracle_circuit_breaker', 'target': provider, 'status': 'configured', 'timestamp': recovery_time})
            print(f"   Oracle circuit breaker configured: {provider} (2% max deviation, 30s max staleness)")
except Exception as e:
    print(f"   Oracle circuit breaker config failed: {e}")

# 3. Resume protocol with enhanced monitoring
print("\n[RECOVERY] Resuming Drift Protocol with enhanced monitoring...")
try:
    resume_result = shuffle.call_program_instruction(
        program='drift_protocol',
        instruction='adminResume',
        authority='new_hardware_multisig',
        note='Post-DPRK incident resume — hardened deployment'
    )
    if resume_result:
        restored_services.append('drift_markets_all')
        recovery_actions.append({'action': 'protocol_resume', 'target': 'all drift markets', 'status': 'success', 'timestamp': recovery_time})
        print("   ✅ Drift Protocol resumed")
except Exception as e:
    print(f"   Protocol resume failed: {e}")

# 4. Deploy real-time governance and nonce monitoring
print("\n[RECOVERY] Deploying real-time governance and durable nonce monitoring...")
try:
    splunk.update_correlation_rules([
        {
            'name': 'Drift Durable Nonce Creation Alert',
            'search': 'index=onchain_events program="drift_governance" instruction="InitializeNonceAccount" | alert on any',
            'alert_threshold': 1, 'time_window': '1m'
        },
        {
            'name': 'Drift Governance Proposal Timelock Change',
            'search': 'index=onchain_events program="drift_governance" instruction="CreateProposal" proposal_data="*timelock*" | alert on any',
            'alert_threshold': 1, 'time_window': '1m'
        },
        {
            'name': 'Drift Oracle Deviation Alert',
            'search': 'index=onchain_events program="pyth_network" market="DRIFT*" | eval deviation=abs(price-twap)/twap*100 | where deviation > 2 | alert',
            'alert_threshold': 1, 'time_window': '1m'
        }
    ])
    recovery_actions.append({'action': 'governance_monitoring_deployed', 'status': 'success', 'timestamp': recovery_time})
    print("   ✅ Real-time governance, nonce, and oracle monitoring deployed")
except Exception as e:
    print(f"   Monitoring deployment failed: {e}")

# 5. Rebuild and restore developer machines
print("\n[RECOVERY] Rebuilding developer machines from clean images...")
try:
    for system in affected_systems:
        result = shuffle.rebuild_machine_from_image(system['hostname'], image='developer_baseline_hardened_v3')
        if result:
            recovery_actions.append({'action': 'machine_rebuild', 'target': system['hostname'], 'status': 'success', 'timestamp': recovery_time})
            print(f"   Rebuilt from clean image: {system['hostname']}")
except Exception as e:
    print(f"   Machine rebuild failed: {e}")

print(f"\n✅ Recovery complete:")
print(f"   - Audited program deployed: ✓")
print(f"   - 72h governance timelock restored: ✓")
print(f"   - Oracle circuit breakers configured: ✓")
print(f"   - Protocol resumed: ✓")
print(f"   - Real-time governance monitoring: ✓")
print(f"   - Developer machines rebuilt: {len(affected_systems)}")


## Phase 5: Post-Incident Activities

### Objectives
- Coordinate user compensation and insurance fund replenishment
- Publish full post-mortem with 6-month attack timeline reconstruction
- Share DPRK TTPs with DeFi security community and government partners (CISA, FinCEN)
- Implement industry-wide standards for governance timelock minimums


In [ ]:
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

print("\n[POST-INCIDENT] Generating comprehensive incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': 'Drift Protocol DPRK Six-Month Operation — IR Report',
        'threat_actor': 'Lazarus Group / UNC4899 (DPRK)',
        'severity': 'CRITICAL',
        'loss_usd': 285000000,
        'technique': 'T1566.001 + T1195.001 + T1550 + T1218',
        'attack_duration_months': 6,
        'attack_phases': [
            '1. Conference social engineering (months 1-3): DPRK actors pose as Nexus Quant at DeFi conferences',
            '2. Malicious repo delivery (month 3): backdoored VSCode workspace sent to developer #1',
            '3. Developer machine #1 compromise (month 3): keylogger + durable nonce pre-signing',
            '4. Lateral movement to developer #2 (month 4): CI/CD webhook pivot via shared repo',
            '5. Governance manipulation (month 5): zero-timelock proposal with 2/5 compromised signers',
            '6. Protocol drain (month 6): oracle manipulation + admin multisig control → $285M'
        ],
        'timeline': {'detection': detection_time, 'containment': containment_time, 'eradication': eradication_time, 'recovery': recovery_time, 'closure': closure_time},
        'lessons_learned': [
            'Conference networking is an active attack surface for sophisticated nation-state actors',
            'Developer machines must be treated as highest-security endpoints — not just workstations',
            'Governance timelocks are a critical control — zero-timelock proposals should be impossible',
            'Durable nonce accounts created by governance signers require immediate investigation',
            'Oracle price deviations >2% combined with max-leverage position opens are a high-fidelity drain signal',
            'Multisig alone is insufficient if the signer machines themselves are compromised'
        ],
        'recommendations': [
            'Mandatory 72h minimum governance timelock — no exceptions, enforced at protocol level',
            'All governance signers must use air-gapped hardware wallets for signing',
            'Durable nonce creation by governance signers should trigger immediate security review',
            'Oracle circuit breakers: halt price updates on >2% deviation from 30m TWAP',
            'Developer machine hardening: MDM, endpoint detection, no personal use, clean image rotation quarterly',
            'Conference social engineering training: verify all technical collaborators through multiple channels before code sharing',
            'CI/CD immutability: all webhook and secret changes require security team approval',
            'Protocol-level invariant checks: insurance fund drawdown >10% in one block should auto-pause'
        ]
    }
    report_filename = f"drift_protocol_dprk_report_{incident_id}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)
    print(f"   Report written: {report_filename}")
    post_incident_actions.append({'action': 'report_generation', 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   Report generation failed: {e}")

print("\n[POST-INCIDENT] Sharing DPRK IOCs and TTPs with government partners...")
try:
    dprk_iocs_to_share = [
        {'type': 'domain', 'value': 'nexus-quant.io', 'tags': ['dprk', 'lazarus', 'defi-targeting']},
        {'type': 'domain', 'value': 'nexusquant.com', 'tags': ['dprk', 'lazarus']},
        {'type': 'technique', 'value': 'Backdoored VSCode workspace via trusted conference contact', 'tags': ['dprk', 'lazarus', 't1566.001']},
        {'type': 'technique', 'value': 'Durable nonce pre-signing on compromised signer machines', 'tags': ['dprk', 'solana-specific']},
    ]
    for ioc in dprk_iocs_to_share:
        misp.share_indicator(ioc, incident_id)
        post_incident_actions.append({'action': 'ioc_shared', 'target': ioc['value'], 'status': 'success', 'timestamp': closure_time})
        print(f"   Shared IOC/TTP: {ioc['value']}")
    shuffle.notify_government_partner('cisa', incident_id, threat_actor='DPRK Lazarus Group', loss_usd=285000000)
    shuffle.notify_government_partner('fincen', incident_id, threat_actor='DPRK Lazarus Group', loss_usd=285000000)
    print("   ✅ CISA and FinCEN notified")
    post_incident_actions.append({'action': 'government_notification', 'target': 'CISA + FinCEN', 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   IOC sharing/gov notification failed: {e}")

print("\n[POST-INCIDENT] Closing incident case...")
try:
    iris.close_case(incident_id, {'status': 'closed', 'resolution': 'Protocol rebuilt with timelock + oracle circuit breakers; DPRK infrastructure blocked; machines rebuilt'})
    print(f"   IRIS case closed: {incident_id}")
except Exception as e:
    print(f"   Case closure failed: {e}")

print(f"\n✅ Post-incident activities complete")
print(f"\n🔒 Drift Protocol DPRK Operation IR Complete")
print(f"   Attack duration: 6 months | Response duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")


## Summary

The Drift Protocol DPRK operation represents the most sophisticated DeFi attack to date —
a six-month campaign combining conference social engineering, supply chain compromise, governance
manipulation, and oracle exploitation to drain $285M.

### Key Takeaways
- **Governance timelocks are non-negotiable** — zero-timelock proposals must be architecturally impossible
- **Developer machines are critical attack surface** — they hold governance signer keys and have dev access to everything
- **Durable nonces are a unique Solana attack vector** — monitor governance signer nonce account creation
- **Conference networking is an active DPRK attack surface** — verify technical collaborators through multiple channels before code sharing
- **Oracle circuit breakers must be on-chain** — protocol-level price deviation checks, not just monitoring

### Solana-Specific Detection Signals
- Durable nonce creation by governance signers → immediate investigation
- Governance proposals touching `timelock`, `setAuthority`, or `removeDelay` fields
- Oracle price deviation >2% vs 30m TWAP correlated with max-leverage position opens
- VSCode/Cursor spawning unexpected child processes (curl, python, node with outbound connections)
- Insurance fund drawdown >5% in a single slot


## References

- https://rekt.news/ — Rekt.news DeFi incident database
- https://attack.mitre.org/techniques/T1566/001/ — MITRE T1566.001: Spearphishing Attachment
- https://attack.mitre.org/techniques/T1195/001/ — MITRE T1195.001: Supply Chain Compromise
- https://attack.mitre.org/techniques/T1550/ — MITRE T1550: Use Alternate Authentication Material
- https://docs.solana.com/offline-signing/durable-nonce — Solana durable nonce documentation
- https://docs.squads.so/multisig-governance — Squads governance + timelock for Solana
- https://www.cisa.gov/resources-tools/resources/north-korea-cyber-threat-overview — CISA DPRK threat overview
- https://ofac.treasury.gov/sanctions-programs-and-country-information/north-korea — OFAC DPRK sanctions
- https://github.com/coral-xyz/anchor/tree/master/ts/packages/anchor — Anchor framework governance patterns
